# Modular pipeline: execution, reload and parity

**Purpose.** Run the merged one-command pipeline (`python -m src.main`), summarise its data, per-asset and macro
metrics and the validation quality gate, reload the saved candidate for raw-candle inference without retraining, and
compare the results with the standalone reference notebook when its outputs are available.
This notebook verifies the agreed fixed choices; it does not tune, select or change anything.

**Prerequisites**
- The team environment (`conda activate crypto-volatility-mlops`, Python 3.12) as the kernel.
- Kernel working directory = repository root. `sys.path` is not modified.
- The 54 monthly Binance CSVs under `data/raw/<SYMBOL>/`, or `ML_RAW_DIR` set to an existing raw-data root
  before starting Jupyter. No data are copied.
- Optional: run `notebooks/crypto_volatility_ml.ipynb` first to enable the standalone comparison (section 5).

**Limits.** The July–September 2026 final period has prior analytical exposure; it is not an untouched holdout and
must not guide tuning. Scores are uncalibrated risk scores. A gate pass leaves the candidate
`candidate_pending_review`; it is not release approval. Generated data, models and reports stay out of Git.

In [1]:
import json
import os
import platform
import subprocess
import sys
import time
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from src import infer, main, utils

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "main.py").is_file():
    raise FileNotFoundError("Start the kernel with the repository root as its working directory.")
RAW_OVERRIDE = os.environ.get("ML_RAW_DIR")
# Same path resolution as `python -m src.main`; all runtime values come from main.SETTINGS.
settings = main.resolved_settings(raw_dir=Path(RAW_OVERRIDE) if RAW_OVERRIDE else None)
PATHS, COLUMNS = settings["paths"], settings["columns"]
MASKS = [(str(PATHS["raw_dir"]), "<raw_dir>"), (str(ROOT), "<repo>")]


def shown(text):
    """Replace machine-specific folders in displayed text with placeholders."""
    for actual, placeholder in MASKS:
        text = str(text).replace(actual, placeholder)
    return text


print(f"Python {platform.python_version()} | " + ", ".join(
    f"{name} {version(name)}" for name in ["pandas", "numpy", "scikit-learn", "joblib"]))
print("Raw data:", "ML_RAW_DIR override" if RAW_OVERRIDE else "data/raw (repository default)")
print("Outputs:", {key: shown(path) for key, path in PATHS.items() if key != "raw_dir"})

Python 3.12.16 | pandas 3.1.1, numpy 2.5.4, scikit-learn 1.9.1, joblib 1.6.0
Raw data: ML_RAW_DIR override
Outputs: {'clean_data': '<repo>/data/processed/clean.csv', 'model': '<repo>/models/model.joblib', 'predictions': '<repo>/reports/predictions.csv', 'metrics': '<repo>/reports/metrics.json'}


## 1. Run the one-command pipeline
The pipeline runs in a fresh Python process exactly as in the terminal, inheriting `ML_RAW_DIR`.
Only the key log lines are shown, with local folders replaced by placeholders.

In [2]:
started = time.perf_counter()
run = subprocess.run([sys.executable, "-m", "src.main"], cwd=ROOT, capture_output=True, text=True)
log = shown(run.stderr).splitlines()
if run.returncode != 0:
    print("\n".join(log[-25:]))
    raise RuntimeError(f"python -m src.main failed with exit code {run.returncode}")
keys = ["Loaded", "Cleaned", "Chronological split", "cutoffs", "Dropped", "Quality gate", "Candidate artifact",
        "Pipeline finished"]
print("\n".join(dict.fromkeys(line for line in log if any(key in line for key in keys))))
print(f"Exit code 0 in {time.perf_counter() - started:.0f} s")
report = json.loads(PATHS["metrics"].read_text())

INFO src.load_data: Loaded 2367360 rows from 54 files in <raw_dir>
INFO src.clean_data: Cleaned candles: kept 2367360 of 2367360 rows
INFO __main__: Chronological split: {'train': 1576578, 'validation': 393078, 'test': 397443}; 84 boundary rows purged
INFO src.target: Volatility cutoffs (training only, quantile 0.90): {'BTCUSDT': 0.003477, 'ETHUSDT': 0.005454, 'SOLUSDT': 0.00592}
INFO src.target: Dropped 45 row(s) without a known future (cannot be labelled)
INFO __main__: Quality gate met: average_precision 0.515145 vs threshold 0.490392 (1.20 x benchmark 0.408660)
INFO __main__: Candidate artifact: <repo>/models/candidates/model.joblib
INFO __main__: Pipeline finished; validation quality gate True; candidate still requires review
Exit code 0 in 22 s


## 2. Data summary
Clean candles written by the pipeline, the chronological split sizes and the training-only label cutoffs.

In [3]:
time_col, symbol_col = COLUMNS["time_col"], COLUMNS["symbol_col"]
candles = pd.read_csv(PATHS["clean_data"], usecols=[symbol_col, time_col])
candles[time_col] = pd.to_datetime(candles[time_col], utc=True)
display(candles.groupby(symbol_col)[time_col].agg(candles="size", first_open="min", last_open="max"))
print("Raw files hashed in the run evidence:", len(report["evidence"]["data_manifest"]))
splits = pd.DataFrame({
    "eligible_rows": [report["rows"][part] for part in ("train", "validation", "test")],
    "positive_rate": [np.nan, report["validation"]["positive_rate"], report["test"]["positive_rate"]],
}, index=["train", "validation", "final (prior exposure)"])
display(splits)
print("Split settings (last decision minute per period, UTC):", settings["split"])
print("Training-only cutoffs (90th percentile of future RV):",
      {symbol: round(value, 6) for symbol, value in report["volatility_cutoffs"].items()})

,candles,first_open,last_open
symbol,,,
BTCUSDT,789120,2025-04-01 00:00:00+00:00,2026-09-30 23:59:00+00:00
ETHUSDT,789120,2025-04-01 00:00:00+00:00,2026-09-30 23:59:00+00:00
SOLUSDT,789120,2025-04-01 00:00:00+00:00,2026-09-30 23:59:00+00:00


Raw files hashed in the run evidence: 54


,eligible_rows,positive_rate
train,1576578,NaN
validation,393078,0.064700
final (prior exposure),397398,0.036885


Split settings (last decision minute per period, UTC): {'train_end': '2026-03-31 23:59:00', 'validation_end': '2026-06-30 23:59:00'}
Training-only cutoffs (90th percentile of future RV): {'BTCUSDT': 0.003477, 'ETHUSDT': 0.005454, 'SOLUSDT': 0.00592}


## 3. Per-asset metrics and the validation quality gate
Macro AP is the equal-weight mean of BTC, ETH and SOL `average_precision`. The benchmark is the 60-candle
realised volatility (59 internal returns) on the same rows. The gate uses validation only:
model macro AP ≥ 1.20 × benchmark macro AP. Final-period values are shown for information only.

In [4]:
symbols = settings["rules"]["allowed_symbols"]
table = {}
for part, label in [("validation", "validation"), ("test", "final")]:
    for source, name in [(part, "model"), (f"benchmark_{part}", "benchmark")]:
        values = report[source]
        table[(label, name)] = [values[f"average_precision_{s}"] for s in symbols] + [values["average_precision"]]
metrics = pd.DataFrame(table, index=[*symbols, "macro"]).round(6)
display(metrics)
gate = report["quality_gate"]
expected = settings["quality_gate"]
assert gate["metric"] == expected["metric"] == "average_precision"
assert gate["min_ratio_vs_benchmark"] == expected["min_ratio_vs_benchmark"] == 1.2
assert np.isclose(gate["threshold"], 1.2 * gate["benchmark"])
assert np.isclose(gate["model"], np.mean([report["validation"][f"average_precision_{s}"] for s in symbols]))
print(f"Quality gate {'PASSED' if gate['passed'] else 'NOT MET'}: model {gate['model']:.6f} vs required "
      f"{gate['threshold']:.6f} (1.20 x benchmark {gate['benchmark']:.6f})")
print("Approval status:", report["approval_status"], "| scores: uncalibrated | not release approval")

validation               final          
             model benchmark     model benchmark
BTCUSDT   0.599682  0.512730  0.509414  0.363260
ETHUSDT   0.442748  0.328590  0.310568  0.223243
SOLUSDT   0.503005  0.384659  0.422859  0.314534
macro     0.515145  0.408660  0.414280  0.300345

Quality gate PASSED: model 0.515145 vs required 0.490392 (1.20 x benchmark 0.408660)
Approval status: candidate_pending_review | scores: uncalibrated | not release approval


## 4. Saved-candidate reload and raw-candle inference (no training)
The candidate bundle is checked against the run's SHA-256, then a **fresh Python process** loads it with
`src.infer.score_candles` and scores exact 60-candle windows taken from the clean candles. Each window is sent
twice: with historical microsecond and REST millisecond opening times. Scores must agree with each other and with
the pipeline's batch `predictions.csv` for the same symbol and opening time. These rows come from the final
period, which has prior analytical exposure; they demonstrate reproducibility, not independent performance.

In [5]:
artifact = Path(report["artifact"])
folder = "candidates" if gate["passed"] else "failed_candidates"
assert artifact.parent.name == folder and artifact.is_file()
assert utils.file_sha256(artifact) == report["artifact_sha256"]
print("Artifact:", shown(artifact), "| checksum matches metrics report")

predictions = pd.read_csv(PATHS["predictions"])
predictions[time_col] = pd.to_datetime(predictions[time_col], utc=True)
# Deterministic sample: first, middle and last scored row of the final period for each asset.
sample = pd.concat([group.iloc[[0, len(group) // 2, -1]] for _, group in predictions.groupby(symbol_col)])
clean = pd.read_csv(PATHS["clean_data"])
clean[time_col] = pd.to_datetime(clean[time_col], utc=True)
window_size = settings["features"]["windows"]["long"]
step = pd.Timedelta(settings["rules"]["frequency"])
requests = []
for row in sample.itertuples(index=False):
    asset = clean[clean[symbol_col].eq(getattr(row, symbol_col))]
    last = getattr(row, time_col)
    window = asset[asset[time_col].between(last - (window_size - 1) * step, last)]
    window = window.rename(columns={COLUMNS["trades_col"]: "trades"})
    window = window[[time_col, symbol_col, "open", "high", "low", "close", "volume", "trades"]]
    nanoseconds = window[time_col].dt.as_unit("ns").astype("int64")  # explicit resolution under pandas 3
    for unit, divisor in [("us", 1_000), ("ms", 1_000_000)]:
        requests.append({"unit": unit, "as_of": (last + step).isoformat(),
                         "rows": window.assign(**{time_col: nanoseconds // divisor}).to_dict("records")})

reload_script = '''import json, sys
import pandas as pd
from src import infer
payload = json.load(sys.stdin)
print(json.dumps([infer.score_candles(pd.DataFrame(r["rows"]), payload["artifact"], r["as_of"], unit=r["unit"])
                  ["risk_score"] for r in payload["requests"]]))'''
fresh = subprocess.run([sys.executable, "-c", reload_script], cwd=ROOT, capture_output=True, text=True,
                       input=json.dumps({"artifact": str(artifact), "requests": requests}), timeout=300)
if fresh.returncode != 0:
    raise RuntimeError(shown(fresh.stderr[-2000:]))
scores = np.array(json.loads(fresh.stdout)).reshape(-1, 2)
parity = sample[[symbol_col, time_col, "probability"]].reset_index(drop=True)
parity["raw_us"], parity["raw_ms"] = scores[:, 0], scores[:, 1]
parity["max_abs_diff"] = (parity[["raw_us", "raw_ms"]].sub(parity.probability, axis=0)).abs().max(axis=1)
display(parity)
np.testing.assert_allclose(parity.raw_us, parity.raw_ms, rtol=0, atol=0)
np.testing.assert_allclose(parity.raw_us, parity.probability, rtol=1e-9, atol=1e-12)
print(f"PASS: fresh-process reload without training; {len(parity)} windows; microsecond == millisecond; "
      "raw-candle scores match batch predictions.")

Artifact: <repo>/models/candidates/model.joblib | checksum matches metrics report


,symbol,open_time,probability,raw_us,raw_ms,max_abs_diff
0,BTCUSDT,2026-06-30 23:59:00+00:00,0.006071,0.006071,0.006071,0.0
1,BTCUSDT,2026-08-15 23:52:00+00:00,0.000178,0.000178,0.000178,0.0
2,BTCUSDT,2026-09-30 23:44:00+00:00,0.006003,0.006003,0.006003,0.0
3,ETHUSDT,2026-06-30 23:59:00+00:00,0.004230,0.004230,0.004230,0.0
4,ETHUSDT,2026-08-15 23:52:00+00:00,0.000258,0.000258,0.000258,0.0
5,ETHUSDT,2026-09-30 23:44:00+00:00,0.003518,0.003518,0.003518,0.0
6,SOLUSDT,2026-06-30 23:59:00+00:00,0.012426,0.012426,0.012426,0.0
7,SOLUSDT,2026-08-15 23:52:00+00:00,0.035662,0.035662,0.035662,0.0
8,SOLUSDT,2026-09-30 23:44:00+00:00,0.006796,0.006796,0.006796,0.0


PASS: fresh-process reload without training; 9 windows; microsecond == millisecond; raw-candle scores match batch predictions.


The same public function rejects malformed requests instead of scoring them. This is a short demonstration;
the automated tests remain the evidence for input rejection.

In [6]:
good = pd.DataFrame(requests[0]["rows"])
as_of = pd.Timestamp(requests[0]["as_of"])
cases = {
    "59 candles (incomplete window)": (good.iloc[1:], as_of),
    "duplicate opening time": (good.assign(**{time_col: good[time_col].where(good.index != 1, good[time_col][0])}),
                               as_of),
    "unordered candles": (good.iloc[::-1], as_of),
    "gap (one candle missing)": (good.drop(index=30), as_of),
    "negative close": (good.assign(close=-1.0), as_of),
    "unsupported symbol": (good.assign(**{symbol_col: "DOGEUSDT"}), as_of),
    "last candle not completed": (good, as_of - step),
}
outcomes = []
for name, (frame, when) in cases.items():
    try:
        infer.score_candles(frame, artifact, when, unit="us")
        outcomes.append((name, "ACCEPTED"))
    except (ValueError, KeyError) as error:
        outcomes.append((name, f"rejected: {error}"))
display(pd.DataFrame(outcomes, columns=["request", "outcome"]).set_index("request"))
assert not any(outcome == "ACCEPTED" for _, outcome in outcomes)

,outcome
request,
59 candles (incomplete window),rejected: Require exactly the configured conse...
duplicate opening time,rejected: Duplicate or unordered opening times
unordered candles,rejected: Duplicate or unordered opening times
gap (one candle missing),rejected: Require exactly the configured conse...
negative close,rejected: Require exactly the configured conse...
unsupported symbol,rejected: Require one supported symbol
last candle not completed,rejected: Last candle is not completed


## 5. Standalone versus modular comparison
Uses only files the standalone notebook writes under `results/metrics/` (git-ignored). The comparison is
reported as unavailable if they are missing; the standalone notebook is never run from here. Equivalence of
data and settings is checked first; any mismatch is reported, not hidden.

In [7]:
reference_dir = ROOT / "results" / "metrics"
needed = ["run_summary.json", "selection_lock.json", "data_manifest.json", "validation_by_asset.csv",
          "final_by_asset.csv", "final_benchmark_by_asset.csv"]
missing = [name for name in needed if not (reference_dir / name).is_file()]
comparison_status = "unavailable"
if missing:
    print("UNAVAILABLE: standalone outputs not found:", missing)
    print("Run notebooks/crypto_volatility_ml.ipynb (same environment and data), then rerun this section.")
else:
    lock = json.loads((reference_dir / "selection_lock.json").read_text())
    reference_files = {Path(item["file"]).name: item["sha256"]
                       for item in json.loads((reference_dir / "data_manifest.json").read_text())}
    model_params = {**settings["training"]["params"], "random_state": settings["training"]["random_state"]}
    bundle = utils.load_model(artifact)
    settings_checks = {
        "same raw files and SHA-256": reference_files == report["evidence"]["data_manifest"],
        "standalone selected HGB": lock["selected_model"] == "Histogram Gradient Boosting",
        "same HGB parameters and seed": all(lock["model_parameters"][k] == v for k, v in model_params.items()),
        "same predictor order": lock["predictor_order"] == bundle["metadata"]["predictor_order"],
        "same label rule (q=0.90, linear, >)": (lock["quantile"], lock["quantile_interpolation"],
                                                lock["label_operator"]) == (0.9, "linear", ">"),
        "same split boundaries": (lock["train_end_exclusive"], lock["validation_end_exclusive"]) == (
            "2026-04-01T00:00:00+00:00", "2026-07-01T00:00:00+00:00"),
    }
    display(pd.Series(settings_checks, name="equivalent"))
    validation_ref = pd.read_csv(reference_dir / "validation_by_asset.csv")
    final_ref = pd.read_csv(reference_dir / "final_by_asset.csv")
    final_bench_ref = pd.read_csv(reference_dir / "final_benchmark_by_asset.csv")
    pick = {"validation model": validation_ref[validation_ref.model.eq(lock["selected_model"])],
            "validation benchmark": validation_ref[validation_ref.model.eq("60-candle RV")],
            "final model": final_ref, "final benchmark": final_bench_ref}
    modular = {"validation model": report["validation"], "validation benchmark": report["benchmark_validation"],
               "final model": report["test"], "final benchmark": report["benchmark_test"]}
    rows = []
    for name, frame in pick.items():
        by_symbol = frame.set_index("symbol")
        for symbol in symbols:
            rows.append((name, symbol, by_symbol.loc[symbol, "average_precision"],
                         modular[name][f"average_precision_{symbol}"]))
        rows.append((name, "macro", by_symbol.average_precision.mean(), modular[name]["average_precision"]))
        if name.endswith("model"):
            rows.append((name, "rows", by_symbol.rows.sum(), report["rows"][name.split()[0].replace("final", "test")]))
    for symbol in symbols:
        rows.append(("training cutoff", symbol, lock["label_cutoffs"][symbol], report["volatility_cutoffs"][symbol]))
    comparison = pd.DataFrame(rows, columns=["quantity", "item", "standalone", "modular"])
    comparison["abs_diff"] = (comparison.standalone - comparison.modular).abs()
    display(comparison.set_index(["quantity", "item"]))
    agree = bool(all(settings_checks.values()) and (comparison.abs_diff <= 1e-12).all())
    comparison_status = "agree" if agree else "DISAGREE"
    print("Standalone vs modular:", "all compared quantities agree within 1e-12" if agree
          else "DISAGREEMENT - see rows above; report it, do not adjust either implementation here")

same raw files and SHA-256             True
standalone selected HGB                True
same HGB parameters and seed           True
same predictor order                   True
same label rule (q=0.90, linear, >)    True
same split boundaries                  True
Name: equivalent, dtype: bool

standalone        modular  abs_diff
quantity             item                                           
validation model     BTCUSDT       0.599682       0.599682       0.0
                     ETHUSDT       0.442748       0.442748       0.0
                     SOLUSDT       0.503005       0.503005       0.0
                     macro         0.515145       0.515145       0.0
                     rows     393078.000000  393078.000000       0.0
validation benchmark BTCUSDT       0.512730       0.512730       0.0
                     ETHUSDT       0.328590       0.328590       0.0
                     SOLUSDT       0.384659       0.384659       0.0
                     macro         0.408660       0.408660       0.0
final model          BTCUSDT       0.509414       0.509414       0.0
                     ETHUSDT       0.310568       0.310568       0.0
                     SOLUSDT       0.422859       0.422859       0.0
                     macro         0.414280       0.414280       0.0
                     rows     397398.000000  397398.000000       0.0
final benchmark      BTCUSDT       0.363260       0.363260       0.0
                     ETHUSDT       0.223243       0.223243       0.0
                     SOLUSDT       0.314534       0.314534       0.0
                     macro         0.300345       0.300345       0.0
training cutoff      BTCUSDT       0.003477       0.003477       0.0
                     ETHUSDT       0.005454       0.005454       0.0
                     SOLUSDT       0.005920       0.005920       0.0

Standalone vs modular: all compared quantities agree within 1e-12


## 6. Status

In [8]:
status = {
    "pipeline_exit_code": run.returncode,
    "validation_gate": "passed" if gate["passed"] else "NOT met",
    "approval_status": report["approval_status"],
    "reload_and_unit_parity": "passed",
    "malformed_requests_rejected": f"{len(outcomes)}/{len(outcomes)}",
    "standalone_comparison": comparison_status,
    "open": ["independent future confirmation", "calibration", "alert/freshness limits",
             "business benefit", "team model review"],
}
print(json.dumps(status, indent=2))

{
  "pipeline_exit_code": 0,
  "validation_gate": "passed",
  "approval_status": "candidate_pending_review",
  "reload_and_unit_parity": "passed",
  "malformed_requests_rejected": "7/7",
  "standalone_comparison": "agree",
  "open": [
    "independent future confirmation",
    "calibration",
    "alert/freshness limits",
    "business benefit",
    "team model review"
  ]
}
